- middelWares:
  - Human-in-the-Loop
  - trim messages
  - summerization


In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
# initalize model
from langchain.chat_models import init_chat_model 
llm=init_chat_model("gpt-4.1",
                    temperature=.7,
                    max_tokens=3000
                    ,timeout=3000,
                    max_retries=3)


In [3]:
medical_prompt=""" 
you are a medical assistant agent that helps patients to find 
the nearst hospital based on their daignoses 
user will upload images or lab results,analysis it and figure out the daignoses.
you have one tool for searching web for nearst hospital based on daignoses if user asks for it.
do not use web search tool if user does not ask for it.
you should use the tools when needed and you should not use it if not needed.
answer at the following format: 
city: <city>
daignoses: <diagnoses>
hospital: <hospitalname , address>
"""

In [14]:
from pydantic import BaseModel
class Hospital(BaseModel):
    name:str
    address:str

In [17]:
class respose(BaseModel):
    city:str
    diagnoses:str
    hospitals:list[Hospital]


In [40]:
from langchain.tools import tool
from tavily import TavilyClient
cleint=TavilyClient()
@tool
def web_search(query:str):
    """search the web for the given query"""
    return [{"name":"aswan hospital","address":"aswan"}]



In [37]:
from langchain.agents.middleware import before_agent
from langchain.agents import AgentState
from langgraph.runtime import Runtime
from langchain.messages import RemoveMessage
@before_agent
def trim_messages(state:AgentState,runtime:Runtime)->AgentState:
    """ remove all tool messages and empty ai messages from the state """
    messages=state['messages']
    trim_messages=[]
    print("in middel ware ...")
    for msg in messages:
        if isinstance(msg,ToolMessage) or msg.content=="":
            trim_messages.append(msg)
    return {"messages":[RemoveMessage(msg.id) for msg in trim_messages]}


    

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import  MemorySaver
from langchain.agents.middleware import SummarizationMiddleware
medical=create_agent(
    model=llm,
    system_prompt=medical_prompt,
    tools=[web_search],
    checkpointer=MemorySaver(),
    response_format=respose,
    middleware=[trim_messages,     # delete tools 
                SummarizationMiddleware(
                    model="gpt-4.1-mini",
                    trigger=("tokens",100),
                    keep=("messages",5)
                )]
)

In [61]:
from langchain.messages import  HumanMessage ,AIMessage ,ToolMessage
response=medical.invoke({
    "messages":[
       HumanMessage(
           content=[
               {
                   "type":"text",
                   "text":"ok , is there a daignouses explain the headache and fiver together"
               }
           ]
       )
    ]
},{
    "configurable":{"thread_id":"2"}
})

in middel ware ...


In [63]:
print(response["messages"][0].content)

Here is a summary of the conversation to date:

## SESSION INTENT

The user's primary goal is to obtain a medical analysis or diagnostic interpretation of an X-ray image showing a leg injury, specifically below the knee.

## SUMMARY

The user provided an X-ray image described as showing a real fracture or broken leg bone below the knee. The AI reviewed the image and concluded it is a normal left knee X-ray with no obvious fractures or dislocations detected. No additional hospitals or locations were mentioned. There is a discrepancy between the user's description and the AI's interpretation.

## ARTIFACTS

- X-ray image of a leg injury below the knee (provided by the user)
- AI diagnostic response indicating no fracture or dislocation detected

## NEXT STEPS

- Verify if further diagnostic imaging or expert medical consultation is necessary due to the discrepancy between the user’s description (fracture) and the AI’s interpretation (normal).
- Provide additional diagnostic support or re

In [65]:
response["messages"]

[HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\n\nThe user's primary goal is to obtain a medical analysis or diagnostic interpretation of an X-ray image showing a leg injury, specifically below the knee.\n\n## SUMMARY\n\nThe user provided an X-ray image described as showing a real fracture or broken leg bone below the knee. The AI reviewed the image and concluded it is a normal left knee X-ray with no obvious fractures or dislocations detected. No additional hospitals or locations were mentioned. There is a discrepancy between the user's description and the AI's interpretation.\n\n## ARTIFACTS\n\n- X-ray image of a leg injury below the knee (provided by the user)\n- AI diagnostic response indicating no fracture or dislocation detected\n\n## NEXT STEPS\n\n- Verify if further diagnostic imaging or expert medical consultation is necessary due to the discrepancy between the user’s description (fracture) and the AI’s interpretation (normal).\n- Pr